# Tome 3, Chapter 2: Discrete Random Variables (Python companion)

Exercises first, solutions at the end. We compute exact laws with fractions where we can, and simulate where we cannot: the cycles of random permutations (Feller's coupling), fixed points against Poisson, the speed of the law of rare events, randomised quicksort, the coupon collector and its Gumbel limit, reservoir sampling, the hundred prisoners and the birthday waiting time.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import random
from fractions import Fraction
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2)
random.seed(2)

## Exercises

**P2.1 (★) Cycles, exactly and by simulation.** Compute the exact law of the number $C_n$ of cycles of a random permutation of $n=10$ elements (a) by enumerating all $10!$ permutations is too slow, so use the recursion $c(n,k)=c(n-1,k-1)+(n-1)c(n-1,k)$ for the number of permutations with $k$ cycles; (b) as the law of $B_1+\dots+B_n$ of Theorem 2.12, by convolution. Check that they agree exactly, then sample $10^5$ permutations with `numpy` and compare. Check that the number of records (Exercise 2.1) has the same law.

In [ ]:
# Your code here


**P2.2 (★) Fixed points against Poisson.** For $n=1,\dots,15$, compute exactly the total variation distance $\frac12\sum_j|P(F_n=j)-e^{-1}/j!|$ between the law of the number of fixed points and the Poisson law of parameter $1$ (Theorem 2.13). How fast does it decrease?

In [ ]:
# Your code here


**P2.3 (★) The law of rare events, quantitatively.** Compute the total variation distance between the binomial$(n,\lambda/n)$ and Poisson$(\lambda)$ laws for $\lambda=3$ and $n=10,\dots,10^4$. Plot it on log-log axes. Compare with fixed points: which approximation is better, and why?

In [ ]:
# Your code here


**P2.4 (★★) Randomised quicksort.** Implement quicksort with uniformly random pivots, counting comparisons. For $n=1000$, run it $5000$ times; compare the mean with $2(n+1)H_n-4n$ (Exercise 2.3) and draw the histogram. Is it symmetric?

In [ ]:
# Your code here


**P2.5 (★★) The coupon collector and the Gumbel law.** Simulate $T_n$ for $n=1000$, $20\,000$ times. Compare the mean and the variance with Theorem 2.14, and the histogram of $(T_n-n\ln n)/n$ with the *Gumbel* density $e^{-x}e^{-e^{-x}}$, the limit found by Erdős and Rényi (1961).

In [ ]:
# Your code here


**P2.6 (★) Reservoir sampling.** Implement the algorithm of Exercise 2.5, run it $150\,000$ times on a stream of length $6$ with $k=2$, and check that the $15$ possible samples are equally frequent.

In [ ]:
# Your code here


**P2.7 (★) The hundred prisoners.** Simulate the cycle-following strategy of Exercise 2.6 and the naive strategy (each prisoner opens $50$ random boxes). Compare with $1-(H_{100}-H_{50})$ and $2^{-100}$.

In [ ]:
# Your code here


**P2.8 (★★) Waiting for a birthday.** Compute $\mathbb E[W_N]=\sum_{n=0}^Nq_n$ exactly (Exercise 2.7) for $N=10,10^2,\dots,10^6$, and compare with $\sqrt{\pi N/2}$. Ramanujan found the next term: $\sqrt{\pi N/2}+\frac23$. Check it.

In [ ]:
# Your code here


---
## Solutions

### P2.1

In [ ]:
n = 10
# (a) unsigned Stirling numbers of the first kind
c = [[0] * (n + 1) for _ in range(n + 1)]
c[0][0] = 1
for m in range(1, n + 1):
    for k in range(1, m + 1):
        c[m][k] = c[m - 1][k - 1] + (m - 1) * c[m - 1][k]
law_a = [Fraction(c[n][k], math.factorial(n)) for k in range(n + 1)]
# (b) convolution of Bernoulli(1/j)
law_b = [Fraction(1)]
for j in range(1, n + 1):
    p = Fraction(1, j)
    new = [Fraction(0)] * (len(law_b) + 1)
    for k, q in enumerate(law_b):
        new[k] += q * (1 - p); new[k + 1] += q * p
    law_b = new
print('exact laws agree:', law_a == law_b[:n + 1])
print('mean', float(sum(k * q for k, q in enumerate(law_a))), ' H_10 =', sum(1 / j for j in range(1, n + 1)))

def n_cycles(perm):
    seen, cnt = np.zeros(len(perm), bool), 0
    for i in range(len(perm)):
        if not seen[i]:
            cnt += 1
            j = i
            while not seen[j]:
                seen[j] = True; j = perm[j]
    return cnt
def n_records(perm):
    return int(np.sum(perm == np.maximum.accumulate(perm)))
m = 100_000
cyc = Counter(n_cycles(rng.permutation(n)) for _ in range(m))
rec = Counter(n_records(rng.permutation(n)) for _ in range(m))
print(' k   exact    cycles   records')
for k in range(1, 8):
    print(f'{k:2d}  {float(law_a[k]):.4f}   {cyc[k]/m:.4f}   {rec[k]/m:.4f}')

The recursion (which counts permutations by where the element $n$ goes: a fixed point, or inserted after one of the $n-1$ others in their cycles) and the convolution of Bernoulli laws give the same fractions, as Feller's coupling predicts. Records have the same law, for a reason that is a bijection in disguise: writing a permutation in a canonical cycle notation where each cycle starts with its *largest* element, cycles in increasing order of their largest elements, and erasing the brackets, turns cycles into left-to-right records (this is *Foata's fundamental bijection*).

### P2.2

In [ ]:
def derangements(m):
    d = [1, 0]
    for k in range(2, m + 1):
        d.append((k - 1) * (d[-1] + d[-2]))
    return d
D = derangements(20)
for n in range(1, 16):
    law = [math.comb(n, j) * D[n - j] / math.factorial(n) for j in range(n + 1)]
    pois = [math.exp(-1) / math.factorial(j) for j in range(n + 1)]
    tv = 0.5 * (sum(abs(a - b) for a, b in zip(law, pois)) + (1 - sum(pois)))
    print(f'n = {n:2d}: TV = {tv:.3e}    2^n/n! = {2**n/math.factorial(n):.3e}')

The distance decreases *superexponentially*, roughly like $2^n/n!$: already at $n=10$ it is below $10^{-4}$. This matches Exercise 2.4: the first $n$ factorial moments of $F_n$ are exactly those of the Poisson law, so the two laws agree to a very high order (the bound $d_{TV}\le2^n/n!$ is due to Diaconis and Pitman; see Arratia and Tavaré, *Annals of Probability* 20, 1992).

### P2.3

In [ ]:
from math import lgamma, log, exp
def binom_pmf(k, n, p):
    return np.exp([lgamma(n + 1) - lgamma(j + 1) - lgamma(n - j + 1) + j * log(p) + (n - j) * log(1 - p) for j in k])
def poisson_pmf(k, lam):
    return np.exp([-lam + j * log(lam) - lgamma(j + 1) for j in k])
lam = 3
ns = np.unique(np.logspace(1, 4, 25).astype(int))
tvs = []
for n in ns:
    k = np.arange(0, n + 1)
    b = binom_pmf(k, n, lam / n)
    p = poisson_pmf(k, lam)
    tvs.append(0.5 * (np.abs(b - p).sum() + (1 - p.sum())))   # the Poisson mass beyond n counts too
slope = np.polyfit(np.log(ns), np.log(tvs), 1)[0]
fig, ax = plt.subplots(figsize=(6, 3))
ax.loglog(ns, tvs, 'o-', ms=3, label=f'TV(binomial, Poisson), slope {slope:.2f}')
ax.loglog(ns, lam**2 / ns, '--', label='$\\lambda^2/n$')
ax.set_xlabel('n'); ax.legend(); plt.show()
print('n = 100:', tvs[list(ns).index(100)] if 100 in ns else None)

The distance decreases like $1/n$ (slope $-1$), below the bound $\lambda^2/n$ (Le Cam's inequality, proved by a coupling in Tome 10). This is much slower than for fixed points: in the binomial case the $n$ trials are truly independent but each has the same small probability, and the error comes from the chance of two successes in the same trial being excluded; for fixed points, the slight negative dependences between the events conspire to make the approximation almost perfect.

### P2.4

In [ ]:
import sys
sys.setrecursionlimit(10000)
def quicksort_comparisons(a):
    if len(a) <= 1: return 0
    pivot = a[random.randrange(len(a))]
    smaller = [x for x in a if x < pivot]
    larger = [x for x in a if x > pivot]
    return len(a) - 1 + quicksort_comparisons(smaller) + quicksort_comparisons(larger)
n = 1000
H = sum(1 / k for k in range(1, n + 1))
runs = [quicksort_comparisons(list(range(n))) for _ in range(5000)]
print(f'mean {np.mean(runs):.1f}, theory {2*(n+1)*H - 4*n:.1f}; sd {np.std(runs):.1f} '
      f'(theory sqrt(7 - 2 pi^2/3) n = {math.sqrt(7 - 2*math.pi**2/3)*n:.1f})')
r = np.array(runs, float)
print('skewness', np.mean((r - r.mean())**3) / r.std()**3)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(runs, bins=60, density=True)
ax.axvline(2*(n+1)*H - 4*n, color='C1', label='$2(n+1)H_n-4n$')
ax.set_xlabel('comparisons'); ax.legend(); plt.show()

The mean matches $2(n+1)H_n-4n\approx10\,986$ for $n=1000$, and the standard deviation is about $0.65\,n$, a classical result of Knuth for the variance, $(7-\frac{2\pi^2}3)n^2+O(n\log n)$. The histogram is visibly skewed to the right: the limit law of $(C_n-\mathbb E C_n)/n$ is not Gaussian (Régnier and Rösler, 1989–1991, who characterised it by a fixed-point equation that mirrors the recursion of the algorithm). Bad runs are rare but heavier than a bell curve would suggest.

### P2.5

In [ ]:
n, m = 1000, 20_000
p = (n - np.arange(n)) / n                          # p_i = (n - i + 1)/n
T = rng.geometric(p, size=(m, n)).sum(axis=1)      # independent geometric phases (Theorem 2.14)
H = sum(1 / k for k in range(1, n + 1))
var = sum((1 - q) / q**2 for q in p)
print(f'mean {T.mean():.0f} (n H_n = {n*H:.0f}); variance {T.var():.3e} (exact {var:.3e} <= pi^2 n^2/6 = {math.pi**2*n**2/6:.3e})')
x = (T - n * math.log(n)) / n
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(x, bins=80, density=True, label='$(T_n-n\\ln n)/n$')
u = np.linspace(-2.5, 7, 300)
ax.plot(u, np.exp(-u) * np.exp(-np.exp(-u)), 'C1', label='Gumbel density')
ax.legend(); plt.show()
for c in [1, 2, 3]:
    print(f'P(T > n ln n + {c} n): {np.mean(x > c):.4f}   bound e^-c = {math.exp(-c):.4f}   Gumbel 1 - exp(-e^-c) = {1 - math.exp(-math.exp(-c)):.4f}')

The variance is close to $\frac{\pi^2}6n^2$, since the bound of Theorem 2.14 loses only the $\sum\frac1{p_i}$ term. The fluctuations of $T_n$ are of order $n$, small compared with the mean $n\ln n$, and their shape is the Gumbel law $P(T_n\le n\ln n+cn)\to e^{-e^{-c}}$. The union bound $e^{-c}$ of Theorem 2.14 is already sharp for large $c$, since $1-e^{-e^{-c}}\approx e^{-c}$.

### P2.6

In [ ]:
def reservoir(stream, k):
    res = list(stream[:k])
    for i in range(k, len(stream)):            # element number i+1
        if random.random() < k / (i + 1):
            res[random.randrange(k)] = stream[i]
    return frozenset(res)
m = 150_000
cnt = Counter(reservoir(list('abcdef'), 2) for _ in range(m))
freqs = np.array(sorted(cnt.values())) / m
print(len(cnt), 'samples seen; frequencies from', freqs.min().round(4), 'to', freqs.max().round(4), '; 1/15 =', round(1/15, 4))
chi2 = sum((v - m / 15) ** 2 / (m / 15) for v in cnt.values())
print(f'chi-square statistic {chi2:.1f} with 14 degrees of freedom (typical value 14 +- 5)')

All $15$ pairs appear with frequency $\frac1{15}$ up to sampling noise; the chi-square statistic is of the size expected under exact uniformity. The algorithm uses $k$ memory cells whatever the length of the stream, which is why it is used to sample from logs and data streams too large to store.

### P2.7

In [ ]:
def prisoners(strategy, n=100):
    boxes = rng.permutation(n)
    for i in range(n):
        if strategy == 'cycle':
            b = i
            for _ in range(n // 2):
                if boxes[b] == i: break
                b = boxes[b]
            else:
                return False
        else:
            if i not in boxes[rng.choice(n, n // 2, replace=False)]:
                return False
    return True
m = 20_000
wins = sum(prisoners('cycle') for _ in range(m))
H = lambda k: sum(1 / j for j in range(1, k + 1))
print(f'cycle strategy: {wins/m:.4f} +- {1.96*math.sqrt(wins/m*(1-wins/m)/m):.4f}; theory {1 - (H(100) - H(50)):.4f}')
print(f'naive strategy: {sum(prisoners("naive") for _ in range(2000))/2000} (theory 2^-100 = {2**-100:.1e})')

Thirty-one percent against $10^{-30}$. Each prisoner still has a probability exactly $\frac12$ of success in both strategies; what changes is the dependence. With cycles, the prisoners succeed or fail together, so the probability that *all* succeed is about the same as that of one.

### P2.8

In [ ]:
def mean_wait(N):
    q, total = 1.0, 0.0
    for n in range(N + 1):
        total += q
        q *= (N - n) / N
    return total
for N in [10, 100, 10**3, 10**4, 10**5, 10**6]:
    e = mean_wait(N)
    a = math.sqrt(math.pi * N / 2)
    print(f'N = {N:>8}: E[W] = {e:12.5f}   sqrt(pi N/2) = {a:12.5f}   + 2/3 = {a + 2/3:12.5f}   difference {e - a - 2/3:+.2e}')
print('birthdays: E[W_365] =', round(mean_wait(365), 3))

The ratio tends to $1$, as Exercise 2.7 proves, and the difference $\mathbb E[W_N]-\sqrt{\pi N/2}$ tends to $\frac23$, with an error of order $1/\sqrt N$. In a room filling up one person at a time, the first shared birthday arrives on average with the $24.6$-th person. (In Ramanujan's notation, $\mathbb E[W_N]=1+Q(N)$, where $Q(N)=\sqrt{\pi N/2}-\frac13+\cdots$, Knuth, *The Art of Computer Programming*, Vol. 1, §1.2.11.3.)